In [1]:
#I chose the following dataset.  My main purpose is to see if coapplicant income makes a difference vs
#loan amount applicant income and credit history https://www.kaggle.com/datasets/krishnaraj30/finance-loan-approval-prediction-data

In [2]:
#import required modules
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
#load dataset
df = pd.read_csv('test.csv')
df2 = df.copy()
print(df.head())

    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001015   Male     Yes          0      Graduate            No   
1  LP001022   Male     Yes          1      Graduate            No   
2  LP001031   Male     Yes          2      Graduate            No   
3  LP001035   Male     Yes          2      Graduate            No   
4  LP001051   Male      No          0  Not Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5720                  0       110.0             360.0   
1             3076               1500       126.0             360.0   
2             5000               1800       208.0             360.0   
3             2340               2546       100.0             360.0   
4             3276                  0        78.0             360.0   

   Credit_History Property_Area  
0             1.0         Urban  
1             1.0         Urban  
2             1.0         Urban  
3             NaN     

In [4]:
#print shape
df.shape

(367, 12)

In [5]:
#check for null values
df.isnull().sum()

Loan_ID               0
Gender               11
Married               0
Dependents           10
Education             0
Self_Employed        23
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            5
Loan_Amount_Term      6
Credit_History       29
Property_Area         0
dtype: int64

In [6]:
#drop null values
df = df.dropna()

In [7]:
#verify
df.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
dtype: int64

In [8]:
#setting variable to determine coapplicant income impact. Testing for greater than 0.
df['Coapp_Diff'] = (df['CoapplicantIncome'] > 0).astype(int)

In [9]:
#drop coapplicantincome and coapp_diff and set y to coapp_diff
x = df.drop(['CoapplicantIncome', 'Coapp_Diff'], axis=1)
y = df['Coapp_Diff']

In [10]:
#train sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.20, random_state = 42)

In [11]:
#set categorical and numeric features
categorical_features = ['Self_Employed', 'Education', 'Gender']
numeric_features = ['ApplicantIncome', 'LoanAmount', 'Credit_History']

In [12]:
#Decision Tree
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [13]:
#decision tree pipeline
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',DecisionTreeClassifier())
])

In [14]:
#accuracy of decision tree
dt_pipeline.fit(x_train, y_train)

y_pred = dt_pipeline.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       0.75      0.67      0.71        27
           1       0.74      0.81      0.77        31

    accuracy                           0.74        58
   macro avg       0.74      0.74      0.74        58
weighted avg       0.74      0.74      0.74        58

Accuracy: 0.741


In [15]:
#random forest
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [16]:
#random forest pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [17]:
#accuracy of random forest
rf_pipeline.fit(x_train, y_train)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       0.75      0.67      0.71        27
           1       0.74      0.81      0.77        31

    accuracy                           0.74        58
   macro avg       0.74      0.74      0.74        58
weighted avg       0.74      0.74      0.74        58

Accuracy: 0.741


In [18]:
#bestparams GridsearchCV
parameters = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_depth': [1, 5, 10],
    'classifier__min_samples_split': [3, 6]
}

grid = GridSearchCV(rf_pipeline, parameters, cv=3, n_jobs=-1)
grid.fit(x_train, y_train)

print("Best Parameters:", grid.best_params_)

y_pred = grid.best_estimator_.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1

Best Parameters: {'classifier__max_depth': 10, 'classifier__min_samples_split': 6, 'classifier__n_estimators': 100}
Accuracy: 0.8275862068965517


In [19]:
#best param pipeline
bp_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        max_depth=10,
        min_samples_split=6,
        n_estimators=100,
        random_state=42))
])

In [20]:
#fit pipeline
bp_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['Self_Employed', 'Education',
                                                   'Gender']),
                                                 ('num', StandardScaler(),
                                                  ['ApplicantIncome',
                                                   'LoanAmount',
                                                   'Credit_History'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=10, min_samples_split=6,
                                        random_state=42))])

In [21]:
#accuracy of best params
y_pred = bp_pipeline.predict(x_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy Score:", accuracy_score(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.85      0.82        27
           1       0.86      0.81      0.83        31

    accuracy                           0.83        58
   macro avg       0.83      0.83      0.83        58
weighted avg       0.83      0.83      0.83        58

Accuracy Score: 0.8275862068965517


In [22]:
#SVC
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [23]:
#svc pipeline
svc_pipeline = Pipeline(steps=[
('preprocessor', preprocessor),
    ('classifier',SVC())
])

In [24]:
#accuracy of SVC
svc_pipeline.fit(x_train, y_train)

y_pred = svc_pipeline.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       0.89      0.59      0.71        27
           1       0.72      0.94      0.82        31

    accuracy                           0.78        58
   macro avg       0.81      0.76      0.76        58
weighted avg       0.80      0.78      0.77        58

Accuracy: 0.776


In [25]:
#setting variable to determine coapplicant income impact. Testing for greater than 2500.
df['Coapp_Diff'] = (df['CoapplicantIncome'] > 2500).astype(int)

In [26]:
#drop coapplicantincome and coapp_diff and set y to coapp_diff
x = df.drop(['CoapplicantIncome', 'Coapp_Diff'], axis=1)
y = df['Coapp_Diff']

In [27]:
#train sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.20, random_state = 42)

In [28]:
#set categorical and numeric features
categorical_features = ['Self_Employed', 'Education', 'Gender']
numeric_features = ['ApplicantIncome', 'LoanAmount', 'Credit_History']

In [29]:
#Decision Tree
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [30]:
#decision tree pipeline
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',DecisionTreeClassifier())
])

In [31]:
#accuracy of decision tree
dt_pipeline.fit(x_train, y_train)

y_pred = dt_pipeline.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       0.85      0.87      0.86        45
           1       0.50      0.46      0.48        13

    accuracy                           0.78        58
   macro avg       0.67      0.66      0.67        58
weighted avg       0.77      0.78      0.77        58

Accuracy: 0.776


In [32]:
#random forest
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [33]:
#random forest pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [34]:
#accuracy of random forest
rf_pipeline.fit(x_train, y_train)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       0.85      0.87      0.86        45
           1       0.50      0.46      0.48        13

    accuracy                           0.78        58
   macro avg       0.67      0.66      0.67        58
weighted avg       0.77      0.78      0.77        58

Accuracy: 0.776


In [35]:
#bestparams GridsearchCV
parameters = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_depth': [1, 5, 10],
    'classifier__min_samples_split': [3, 6]
}

grid = GridSearchCV(rf_pipeline, parameters, cv=3, n_jobs=-1)
grid.fit(x_train, y_train)

print("Best Parameters:", grid.best_params_)

y_pred = grid.best_estimator_.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Best Parameters: {'classifier__max_depth': 5, 'classifier__min_samples_split': 6, 'classifier__n_estimators': 100}
Accuracy: 0.8275862068965517


In [36]:
#best params pipeline
bp_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        max_depth=5,
        min_samples_split=6,
        n_estimators=100,
        random_state=42))
])

In [37]:
#fit pipeline
bp_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['Self_Employed', 'Education',
                                                   'Gender']),
                                                 ('num', StandardScaler(),
                                                  ['ApplicantIncome',
                                                   'LoanAmount',
                                                   'Credit_History'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=5, min_samples_split=6,
                                        random_state=42))])

In [38]:
#accuracy of best params
y_pred = bp_pipeline.predict(x_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy Score:", accuracy_score(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.98      0.90        45
           1       0.80      0.31      0.44        13

    accuracy                           0.83        58
   macro avg       0.82      0.64      0.67        58
weighted avg       0.82      0.83      0.80        58

Accuracy Score: 0.8275862068965517


In [39]:
#SVC
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [40]:
#svc pipeline
svc_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',SVC())
])

In [41]:
#accuracy of SVC
svc_pipeline.fit(x_train, y_train)

y_pred = svc_pipeline.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       0.79      0.98      0.87        45
           1       0.50      0.08      0.13        13

    accuracy                           0.78        58
   macro avg       0.64      0.53      0.50        58
weighted avg       0.72      0.78      0.71        58

Accuracy: 0.776


In [42]:
#setting variable to determine coapplicant income impact. Testing for greater than 4500.
df['Coapp_Diff'] = (df['CoapplicantIncome'] > 4500).astype(int)

In [43]:
#drop coapplicantincome and coapp_diff and set y to coapp_diff
x = df.drop(['CoapplicantIncome', 'Coapp_Diff'], axis=1)
y = df['Coapp_Diff']

In [44]:
#train sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.20, random_state = 42)

In [45]:
#set categorical and numeric features
categorical_features = ['Self_Employed', 'Education', 'Gender']
numeric_features = ['ApplicantIncome', 'LoanAmount', 'Credit_History']

In [46]:
#Decision Tree
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [47]:
#decision tree pipeline
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',DecisionTreeClassifier())
])

In [48]:
#accuracy of decision tree
dt_pipeline.fit(x_train, y_train)

y_pred = dt_pipeline.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       1.00      0.96      0.98        57
           1       0.33      1.00      0.50         1

    accuracy                           0.97        58
   macro avg       0.67      0.98      0.74        58
weighted avg       0.99      0.97      0.97        58

Accuracy: 0.966


In [49]:
#random forest
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [50]:
#random forest pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [51]:
#accuracy of random forest
rf_pipeline.fit(x_train, y_train)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       1.00      0.96      0.98        57
           1       0.33      1.00      0.50         1

    accuracy                           0.97        58
   macro avg       0.67      0.98      0.74        58
weighted avg       0.99      0.97      0.97        58

Accuracy: 0.966


In [52]:
#bestparams GridsearchCV
parameters = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_depth': [1, 5, 10],
    'classifier__min_samples_split': [3, 6]
}

grid = GridSearchCV(rf_pipeline, parameters, cv=3, n_jobs=-1)
grid.fit(x_train, y_train)

print("Best Parameters:", grid.best_params_)

y_pred = grid.best_estimator_.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Best Parameters: {'classifier__max_depth': 1, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 100}
Accuracy: 0.9827586206896551


In [53]:
#best param pipeline
bp_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        max_depth=1,
        min_samples_split=3,
        n_estimators=100,
        random_state=42))
])

In [54]:
#fit pipeline
bp_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['Self_Employed', 'Education',
                                                   'Gender']),
                                                 ('num', StandardScaler(),
                                                  ['ApplicantIncome',
                                                   'LoanAmount',
                                                   'Credit_History'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=1, min_samples_split=3,
                                        random_state=42))])

In [55]:
#accuracy of best params
y_pred = bp_pipeline.predict(x_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy Score:", accuracy_score(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        57
           1       0.00      0.00      0.00         1

    accuracy                           0.98        58
   macro avg       0.49      0.50      0.50        58
weighted avg       0.97      0.98      0.97        58

Accuracy Score: 0.9827586206896551


/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [56]:
#SVC
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [57]:
#svc_pipeline
svc_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',SVC())
])

In [58]:
#accuracy of SVC
svc_pipeline.fit(x_train, y_train)

y_pred = svc_pipeline.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test,y_pred))

print(f"Accuracy: {accuracy:.3f}")

              precision    recall  f1-score   support

           0       0.98      1.00      0.99        57
           1       0.00      0.00      0.00         1

    accuracy                           0.98        58
   macro avg       0.49      0.50      0.50        58
weighted avg       0.97      0.98      0.97        58

Accuracy: 0.983


/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/scottmorel/opt/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [59]:
#In conclusion, there is a strong postive effect of higher coapplicant income in relation to loan predicitons. At 
#Coapp Income between $1 and 2500, accuracy for decision tree and random forest was .741, .827 for GridSearchCV
#best params, and .776 for SVC.  For $2501-4500, it was .776 for decision tree and random forest, same GridSearchCV
#score, and same SVC score.  For $4500 and up, decision tree and random forest were both .966, GridsearchCV was .982
#and SVC was .983.